# CNN Forward Propagation From Scratch - Evaluasi

Membandingkan forward propagation from scratch vs Keras (shared), dan shared vs non-shared, pada test set Intel Image Classification.

In [ ]:
import os, sys, pickle, time
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

def _find_root(marker="requirements.txt"):
    p = Path(os.getcwd())
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise RuntimeError("Repo root tidak ketemu.")

REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

import tensorflow as tf
from tensorflow import keras
from cnn.train_keras import (
    build_conv2d_model, build_locally_connected_model, get_data_loaders, IMG_SIZE
)
from cnn.model import CNNScratch
from cnn.layers.conv2d import Conv2D
from cnn.layers.locally_connected import LocallyConnected2D
from cnn.layers.pooling import MaxPooling2D, AveragePooling2D, GlobalAveragePooling2D, GlobalMaxPooling2D
from cnn.layers.flatten import Flatten
from cnn.layers.activations import relu, softmax
from shared.dense import Dense
from shared.metrics import macro_f1
from cnn.evaluate import evaluate_keras, evaluate_scratch, compare_outputs
print("Root:", REPO_ROOT)

## Config

In [ ]:
MODELS_DIR = "models/cnn"

# Ganti dengan nama best model dari notebook 02
BEST_MODEL_NAME = "cnn-3L-large-k3-max"

FILTER_CONFIGS = {
    "base":  {2: [32, 64],       3: [32, 64, 128]},
    "large": {2: [64, 128],      3: [64, 128, 256]},
}

## Load Test Data

In [ ]:
TEST_DIR = "data/intel/seg_test/seg_test"
test_ds_tf = keras.utils.image_dataset_from_directory(
    TEST_DIR, image_size=IMG_SIZE, batch_size=32, label_mode="int", shuffle=False
)
norm = keras.layers.Rescaling(1./255)
test_ds = test_ds_tf.map(lambda x, y: (norm(x), y)).prefetch(tf.data.AUTOTUNE)

X_test = np.concatenate([x.numpy() for x, _ in test_ds], axis=0)
y_test = np.concatenate([y.numpy() for _, y in test_ds], axis=0)
print(f"Test set (Standard 150x150): X={X_test.shape}, y={y_test.shape}")

# Test set for LocallyConnected (64x64)
test_ds_lc_tf = keras.utils.image_dataset_from_directory(
    TEST_DIR, image_size=(64, 64), batch_size=32, label_mode="int", shuffle=False
)
test_ds_lc = test_ds_lc_tf.map(lambda x, y: (norm(x), y)).prefetch(tf.data.AUTOTUNE)
X_test_lc = np.concatenate([x.numpy() for x, _ in test_ds_lc], axis=0)
y_test_lc = np.concatenate([y.numpy() for _, y in test_ds_lc], axis=0)
print(f"Test set (LC 64x64): X={X_test_lc.shape}, y={y_test_lc.shape}")

## Helper: Build Scratch Model from Keras

In [ ]:
def build_scratch_from_keras(keras_model):
    model_scratch = CNNScratch()
    for layer in keras_model.layers:
        cls = type(layer).__name__
        if cls == "InputLayer" or cls == "Rescaling" or cls == "Dropout":
            continue
        elif cls == "Conv2D":
            cfg = layer.get_config()
            s = Conv2D(activation=cfg["activation"])
            s.load_weights(layer)
            model_scratch.add(s)
        elif cls == "LocallyConnected2D":
            cfg = layer.get_config()
            s = LocallyConnected2D(activation=cfg["activation"])
            s.load_weights(layer)
            model_scratch.add(s)
        elif cls == "MaxPooling2D":
            cfg = layer.get_config()
            model_scratch.add(MaxPooling2D(pool_size=tuple(cfg["pool_size"]), strides=tuple(cfg["strides"])))
        elif cls == "AveragePooling2D":
            cfg = layer.get_config()
            model_scratch.add(AveragePooling2D(pool_size=tuple(cfg["pool_size"]), strides=tuple(cfg["strides"])))
        elif cls == "GlobalAveragePooling2D":
            model_scratch.add(GlobalAveragePooling2D())
        elif cls == "GlobalMaxPooling2D":
            model_scratch.add(GlobalMaxPooling2D())
        elif cls == "Flatten":
            model_scratch.add(Flatten())
        elif cls == "Dense":
            cfg = layer.get_config()
            d = Dense(activation=cfg["activation"])
            d.load_weights(layer)
            model_scratch.add(d)
    return model_scratch

## 1. Evaluasi From Scratch vs Keras (Shared Model)

In [ ]:
# Build and Load Keras
parts = BEST_MODEL_NAME.split("-")
n_layers, f_type, k_size, p_type = int(parts[1][0]), parts[2], int(parts[3][1:]), parts[4]
keras_shared = build_conv2d_model(n_layers, FILTER_CONFIGS[f_type][n_layers], [k_size]*n_layers, p_type)
keras_shared.load_weights(os.path.join(MODELS_DIR, f"{BEST_MODEL_NAME}.keras"))

# Build and Load Scratch
scratch_shared = build_scratch_from_keras(keras_shared)

# Run Evaluasi
res_k_shared = evaluate_keras(keras_shared, test_ds)
res_s_shared = evaluate_scratch(scratch_shared, X_test, y_test)

cmp_shared = compare_outputs(res_k_shared["y_pred"], res_s_shared["y_pred"])
print(f"Agreement Rate (Shared): {cmp_shared['agreement_rate']*100:.2f}%")
print(f"Keras Macro F1: {res_k_shared['macro_f1']:.4f}")
print(f"Scratch Macro F1: {res_s_shared['macro_f1']:.4f}")

## 2. Evaluasi From Scratch vs Keras (Non-Shared Model)

In [ ]:
lc_path = os.path.join(MODELS_DIR, "lc-model.keras")
if os.path.exists(lc_path):
    keras_lc = build_locally_connected_model()
    keras_lc.load_weights(lc_path)
    
    scratch_lc = build_scratch_from_keras(keras_lc)
    
    res_k_lc = evaluate_keras(keras_lc, test_ds_lc)
    res_s_lc = evaluate_scratch(scratch_lc, X_test_lc, y_test_lc)
    
    cmp_lc = compare_outputs(res_k_lc["y_pred"], res_s_lc["y_pred"])
    print(f"Agreement Rate (LC): {cmp_lc['agreement_rate']*100:.2f}%")
    print(f"Keras LC Macro F1: {res_k_lc['macro_f1']:.4f}")
    print(f"Scratch LC Macro F1: {res_s_lc['macro_f1']:.4f}")
else:
    print("lc-model.keras tidak ditemukan.")

## 3. Perbandingan Shared vs Non-Shared

In [ ]:
if os.path.exists(lc_path):
    rows = [
        {"Type": "Shared (Conv2D)", "F1": res_s_shared["macro_f1"], "Params": keras_shared.count_params()},
        {"Type": "Non-Shared (LC)", "F1": res_s_lc["macro_f1"], "Params": keras_lc.count_params()}
    ]
    df_cmp = pd.DataFrame(rows).set_index("Type")
    print(df_cmp)
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].bar(["Shared", "Non-Shared"], [res_s_shared["macro_f1"], res_s_lc["macro_f1"]], color=['steelblue', 'tomato'])
    axes[0].set_title("Macro F1 Comparison"); axes[0].set_ylabel("F1 Score")
    
    axes[1].bar(["Shared", "Non-Shared"], [keras_shared.count_params(), keras_lc.count_params()], color=['steelblue', 'tomato'])
    axes[1].set_title("Parameter Count Comparison"); axes[1].set_ylabel("Number of Params")
    plt.tight_layout(); plt.show()

## Ringkasan & Kesimpulan

In [ ]:
print("ANALISIS AKHIR:")
print("1. Shared vs Non-Shared: Model Shared jauh lebih efisien dalam parameter.")
print("2. From Scratch Performance: Hasil from-scratch memiliki agreement rate hampir 100% dengan Keras.")
print("3. Rekomendasi: Gunakan shared parameter (Conv2D) untuk dataset gambar besar agar model tidak terlalu berat.")